In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/ml_benchmark/06_santander_customer/split_train.csv'
eval_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/ml_benchmark/06_santander_customer/split_eval.csv'

train_df = pd.read_csv(train_data_path)
eval_df = pd.read_csv(eval_data_path)

# Display basic information about the datasets
print("Train Data Info:")
print(train_df.info())
print("\nEval Data Info:")
print(eval_df.info())

# Check for missing values
print("\nMissing values in Train Data:")
print(train_df.isnull().sum())
print("\nMissing values in Eval Data:")
print(eval_df.isnull().sum())

# Distinguish column types
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

# Display basic statistics for numeric columns
print("\nNumeric Columns Statistics:")
print(train_df[numeric_cols].describe())

# Display value counts for categorical columns
print("\nCategorical Columns Value Counts:")
for col in categorical_cols:
    print(f"\n{col}:")
    print(train_df[col].value_counts())

# Visualize the distribution of the target variable
plt.figure(figsize=(8, 6))
sns.countplot(x='target', data=train_df)
plt.title('Distribution of Target Variable')
plt.show()

# Visualize the correlation matrix for numeric columns
plt.figure(figsize=(12, 10))
correlation_matrix = train_df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# Visualize the distribution of a few numeric features
plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols[:5], 1):
    plt.subplot(2, 3, i)
    sns.histplot(train_df[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()


Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160000 entries, 0 to 159999
Data columns (total 19 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   target  160000 non-null  int64  
 1   var_0   160000 non-null  float64
 2   var_1   160000 non-null  float64
 3   var_2   160000 non-null  float64
 4   var_3   160000 non-null  float64
 5   var_4   160000 non-null  float64
 6   var_5   160000 non-null  float64
 7   var_6   160000 non-null  float64
 8   var_7   160000 non-null  float64
 9   var_8   160000 non-null  float64
 10  var_9   160000 non-null  float64
 11  var_10  160000 non-null  float64
 12  var_11  160000 non-null  float64
 13  var_12  160000 non-null  float64
 14  var_13  160000 non-null  float64
 15  var_14  160000 non-null  float64
 16  var_15  160000 non-null  float64
 17  var_16  160000 non-null  float64
 18  var_17  160000 non-null  float64
dtypes: float64(18), int64(1)
memory usage: 23.2 MB
None

Eval Data Info:


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-10 04:02:29.125 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['target', 'var_0', 'var_1', 'var_2', 'var_3', 'var_4', 'var_5', 'var_6', 'var_7', 'var_8', 'var_9', 'var_10', 'var_11', 'var_12', 'var_13', 'var_14', 'var_15', 'var_16', 'var_17'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale

# Copy the DataFrames before processing
train_df_copy = train_df.copy()
eval_df_copy = eval_df.copy()

# Handle missing values
numeric_cols = train_df_copy.select_dtypes(include=[np.number]).columns.tolist()
fill_missing = FillMissingValue(features=numeric_cols, strategy='mean')
train_df_copy = fill_missing.fit_transform(train_df_copy)
eval_df_copy = fill_missing.transform(eval_df_copy)

# Scale numerical features
standard_scale = StandardScale(features=numeric_cols)
train_df_copy = standard_scale.fit_transform(train_df_copy)
eval_df_copy = standard_scale.transform(eval_df_copy)

# Display the first few rows of the processed DataFrames
print("Processed Train Data:")
print(train_df_copy.head())
print("\nProcessed Eval Data:")
print(eval_df_copy.head())


Processed Train Data:
     target     var_0     var_1  ...    var_15    var_16    var_17
0 -0.333657  0.646003  1.127083  ...  1.217056 -1.751116  0.342869
1 -0.333657 -1.693697 -0.261292  ... -1.494532 -1.223731  1.833321
2 -0.333657 -0.425317  0.742922  ... -0.803466 -2.160930 -0.764460
3 -0.333657  0.276557 -1.021842  ...  0.708041 -0.298102  0.734912
4 -0.333657  0.649787 -1.085959  ... -0.328435  0.828507  0.916906

[5 rows x 19 columns]

Processed Eval Data:
     target     var_0     var_1  ...    var_15    var_16    var_17
0  2.997087 -0.662291 -0.103380  ... -0.629426  1.037421 -1.862914
1  2.997087  2.502607  0.379510  ... -0.887938  0.592390  0.443047
2 -0.333657  1.057598  0.303822  ...  0.980389 -0.100367 -1.260925
3 -0.333657 -0.798941  0.385874  ...  0.027899 -2.341741  1.216366
4 -0.333657 -0.353652  1.078952  ... -1.514679  0.430848 -0.952976

[5 rows x 19 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['target', 'var_0', 'var_1', 'var_2', 'var_3', 'var_4', 'var_5', 'var_6', 'var_7', 'var_8', 'var_9', 'var_10', 'var_11', 'var_12', 'var_13', 'var_14', 'var_15', 'var_16', 'var_17'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Assuming train_df_copy and eval_df_copy are already defined and preprocessed
X_train = train_df_copy.drop('target', axis=1)
y_train = train_df_copy['target']
X_eval = eval_df_copy.drop('target', axis=1)
y_eval = eval_df_copy['target']

# Splitting the training data into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Initialize the XGBoost classifier with suitable hyperparameters
xgb_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='auc'
)

# Train the model
xgb_model.fit(
    X_train_split, y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    early_stopping_rounds=50,
    verbose=True
)

# Predict probabilities on the eval set
y_eval_pred_proba = xgb_model.predict_proba(X_eval)[:, 1]

# Calculate AUC on the eval set
auc_score = roc_auc_score(y_eval, y_eval_pred_proba)
print(f"AUC on eval data: {auc_score}")


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'